<a href="https://colab.research.google.com/github/Mugiraneza-user/INDUSTRIAL-ENERGY-SIMULATION-SYSTEM/blob/main/Class_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import numpy as np
import matplotlib.pyplot as plt


system_type = widgets.RadioButtons(
    options=['Grid Only', 'Solar Only', 'Hybrid (Grid + Solar)'],
    value='Grid Only',
    description='System:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)

grid_power = widgets.FloatSlider(
    value=15,
    min=0,
    max=100,
    step=1,
    description='Grid Power (kW):',
    style={'description_width': '150px'}
)

grid_price = widgets.FloatSlider(
    value=133,
    min=0,
    max=500,
    step=10,
    description='Grid Price (FRW/kWh):',
    style={'description_width': '150px'}
)

solar_power = widgets.FloatSlider(
    value=50,
    min=0,
    max=200,
    step=5,
    description='Solar Power (kW):',
    style={'description_width': '150px'}
)

outage_prob = widgets.FloatSlider(
    value=20,
    min=0,
    max=100,
    step=5,
    description='Outage Probability (%):',
    style={'description_width': '150px'}
)

sim_hours = widgets.IntSlider(
    value=24,
    min=1,
    max=168,
    step=1,
    description='Simulation Hours:',
    style={'description_width': '150px'}
)


# MACHINES SECTION


machines = []
machine_container = widgets.VBox()

def add_machine(b):

    machine_name = widgets.Text(
        value=f'Machine_{len(machines)+1}',
        description='Name:',
        layout=widgets.Layout(width='220px')
    )

    machine_power = widgets.FloatSlider(
        value=5,
        min=0,
        max=50,
        step=1,
        description='Power (kW):',
        layout=widgets.Layout(width='350px')
    )

    machine_schedule = widgets.Dropdown(
        options=['24x7', 'daytime', 'business_hours', 'night_shift'],
        value='24x7',
        description='Schedule:',
        layout=widgets.Layout(width='260px')
    )

    remove_btn = widgets.Button(
        description='Remove',
        button_style='danger',
        layout=widgets.Layout(width='90px')
    )

    machine_box = widgets.HBox([
        machine_name,
        machine_power,
        machine_schedule,
        remove_btn
    ])

    def remove_machine(b):
        machines.remove(machine_box)
        machine_container.children = machines

    remove_btn.on_click(remove_machine)

    machines.append(machine_box)
    machine_container.children = machines

add_machine_btn = widgets.Button(
    description='+ Add Machine',
    button_style='success'
)

add_machine_btn.on_click(add_machine)


# RUN BUTTON


run_btn = widgets.Button(
    description='▶ RUN SIMULATION',
    button_style='primary',
    layout=widgets.Layout(
        width='100%',
        padding='25px',
        margin='20px 0px'
    )
)

run_btn.style.font_size = '22px'

# OUTPUT AREA


output = widgets.Output()


# SIMULATION FUNCTION


def run_simulation(b):

    with output:

        clear_output(wait=True)


        # GET CONFIGURATION


        sys_type = system_type.value

        if sys_type == 'Grid Only':
            scenario = 'grid'

        elif sys_type == 'Solar Only':
            scenario = 'solar'

        else:
            scenario = 'hybrid'

        grid_max = grid_power.value
        grid_cost = grid_price.value
        solar_cap = solar_power.value
        outage = outage_prob.value / 100
        hours = sim_hours.value


        # GET MACHINES


        machine_list = []

        for m in machines:

            machine_list.append({
                'name': m.children[0].value,
                'power_kw': m.children[1].value,
                'schedule_type': m.children[2].value
            })

        if len(machine_list) == 0:
            print("Please add at least one machine!")
            return
        # DISPLAY CONFIGURATION



        print(" INDUSTRIAL ENERGY SIMULATION")


        print("\nSYSTEM CONFIGURATION")


        print(f"• System Type: {sys_type}")
        print(f"• Grid Capacity: {grid_max} kW")
        print(f"• Solar Capacity: {solar_cap} kW")
        print(f"• Grid Cost: {grid_cost} FRW/kWh")
        print(f"• Outage Probability: {outage*100}%")
        print(f"• Simulation Hours: {hours}")

        print("\nMACHINES")


        for machine in machine_list:
            print(
                f"• {machine['name']} "
                f"({machine['power_kw']} kW | "
                f"{machine['schedule_type']})"
            )

        # SCHEDULE FUNCTION


        def is_machine_on(schedule_type, t):

            hour = t % 24

            if schedule_type == '24x7':
                return 1

            elif schedule_type == 'daytime':
                return 1 if 6 <= hour <= 18 else 0

            elif schedule_type == 'business_hours':
                return 1 if 8 <= hour <= 18 else 0

            elif schedule_type == 'night_shift':
                return 1 if hour >= 18 or hour <= 6 else 0

            return 0


        # STORAGE VARIABLES

        np.random.seed(42)

        total_cost = 0
        total_demand = 0
        total_unmet = 0
        total_grid = 0
        total_solar = 0

        time_data = []
        demand_data = []
        solar_data = []
        grid_data = []
        unmet_data = []
        cost_data = []


        # MAIN SIMULATION LOOP


        for t in range(hours):

            hour = t % 24


            # MACHINE DEMAND


            demand = 0

            for machine in machine_list:

                if is_machine_on(
                    machine['schedule_type'],
                    hour
                ) == 1:

                    demand += machine['power_kw']

            # SOLAR GENERATION


            if 6 <= hour <= 18:

                angle = np.pi * (hour - 6) / 12
                solar = solar_cap * np.sin(angle)

            else:
                solar = 0


            # GRID STATUS


            grid_available = np.random.random() > outage


            # GRID ONLY


            if scenario == 'grid':

                if grid_available:

                    supply = min(demand, grid_max)
                    grid_used = supply
                    solar_used = 0
                    unmet = demand - supply
                    cost = supply * grid_cost

                else:

                    supply = 0
                    grid_used = 0
                    solar_used = 0
                    unmet = demand
                    cost = 0


            # SOLAR ONLY


            elif scenario == 'solar':

                supply = min(demand, solar)

                grid_used = 0
                solar_used = supply

                unmet = demand - supply
                cost = 0


            # HYBRID SYSTEM


            else:

                remaining = demand

                solar_used = min(remaining, solar)
                remaining -= solar_used

                if grid_available and remaining > 0:

                    grid_used = min(remaining, grid_max)
                    remaining -= grid_used

                else:
                    grid_used = 0

                supply = solar_used + grid_used

                unmet = remaining

                cost = grid_used * grid_cost


            # ACCUMULATE TOTALS


            total_cost += cost
            total_demand += demand
            total_unmet += unmet
            total_grid += grid_used
            total_solar += solar_used


            # STORE GRAPH DATA


            time_data.append(t)
            demand_data.append(demand)
            solar_data.append(solar_used)
            grid_data.append(grid_used)
            unmet_data.append(unmet)
            cost_data.append(cost)


        # KPI CALCULATIONS


        reliability = (
            (1 - total_unmet / total_demand) * 100
            if total_demand > 0 else 100
        )

        renewable_pct = (
            (total_solar / total_demand) * 100
            if total_demand > 0 else 0
        )

        cost_per_kwh = (
             total_cost / total_demand if total_demand > 0 else 0
        )




        print("\n")

        print("SIMULATION RESULTS")


        print("\nCOST ANALYSIS")


        print(f"• Total Cost: {total_cost:,.0f} FRW")
        print(f"• Cost per kWh: {cost_per_kwh:.2f} FRW/kWh")

        print("\nENERGY ANALYSIS")


        print(f"• Total Demand: {total_demand:.1f} kWh")
        print(f"• Grid Energy: {total_grid:.1f} kWh")
        print(f"• Solar Energy: {total_solar:.1f} kWh")
        print(f"• Unmet Energy: {total_unmet:.1f} kWh")

        print("\nPERFORMANCE METRICS")


        print(f"• Reliability: {reliability:.1f}%")
        print(f"• Renewable Share: {renewable_pct:.1f}%")




        # ENERGY FLOW GRAPH


        plt.figure(figsize=(14, 6))

        plt.plot(
            time_data,
            demand_data,
            linewidth=3,
            label='Demand'
        )

        plt.plot(
            time_data,
            solar_data,
            linewidth=3,
            label='Solar Supply'
        )

        plt.plot(
            time_data,
            grid_data,
            linewidth=3,
            label='Grid Supply'
        )

        plt.title('Energy Demand vs Supply')
        plt.xlabel('Simulation Hour')
        plt.ylabel('Energy (kWh)')
        plt.grid(True)
        plt.legend()

        plt.show()


        # UNMET ENERGY GRAPH


        plt.figure(figsize=(14, 5))

        plt.bar(
            time_data,
            unmet_data
        )

        plt.title('Unmet Energy Across Simulation')
        plt.xlabel('Simulation Hour')
        plt.ylabel('Unmet Energy (kWh)')
        plt.grid(True)

        plt.show()



        # ENERGY CONTRIBUTION PIE CHART


        labels = ['Grid Energy', 'Solar Energy']
        sizes = [total_grid, total_solar]

        plt.figure(figsize=(7, 7))

        plt.pie(
            sizes,
            labels=labels,
            autopct='%1.1f%%'
        )

        plt.title('Energy Source Contribution')

        plt.show()




        # FINAL RECOMMENDATION


        print("\n")

        print("SYSTEM CONCLUSION")


        if scenario == 'grid':

            print("• Grid system depends entirely on utility supply.")
            print("• Higher operational cost observed.")
            print("• Vulnerable to outages.")

        elif scenario == 'solar':

            print("• Solar system achieved zero energy cost.")
            print("• Reliability limited during nighttime.")
            print("• Battery storage recommended.")

        else:

            print("• Hybrid system provides best overall balance.")
            print("• Improved reliability and renewable integration.")
            print("• Recommended for medium-scale data centers.")

        print("\nSIMULATION COMPLETE!")



run_btn.on_click(run_simulation)


display(HTML("""
<style>

.jupyter-widgets .p-TabBar-tab {
    font-size: 18px !important;
    font-weight: bold !important;

    background: linear-gradient(135deg, #1e3c72, #2a5298) !important;
    color: white !important;

    min-width: 260px !important;
    width: auto !important;

    border-radius: 12px 12px 0px 0px !important;

    padding: 18px 25px !important;

    margin-right: 6px !important;

    border: none !important;

    transition: all 0.3s ease-in-out !important;
}

.jupyter-widgets .p-TabBar-tab:hover {

    transform: translateY(-3px) scale(1.03);

}

.jupyter-widgets .p-TabBar-tab.p-mod-current {

    background: linear-gradient(
        135deg,
        #11998e,
        #38ef7d
    ) !important;

    color: black !important;

    box-shadow: 0px 4px 12px rgba(0,0,0,0.25) !important;
}

</style>
"""))



tab1 = widgets.VBox([
    widgets.HTML("<h2> ENERGY CONFIGURATION</h2>"),
    system_type,
    grid_power,
    grid_price,
    solar_power,
    outage_prob,
    sim_hours
])

tab2 = widgets.VBox([
    widgets.HTML("<h2> MACHINES CONFIGURATION</h2>"),
    machine_container,
    add_machine_btn
])

tabs = widgets.Tab([tab1, tab2])

tabs.set_title(0, 'Energy Settings')
tabs.set_title(1, 'Machines')


print(" ENERGY SIMULATION SYSTEM")

display(tabs)

display(run_btn)

display(output)

 ENERGY SIMULATION SYSTEM


Button(button_style='primary', description='▶ RUN SIMULATION', layout=Layout(margin='20px 0px', padding='25px'…

Output()

In [ ]:
! git clone https://github.com/Mugiraneza-user/INDUSTRIAL-ENERGY-SIMULATION-SYSTEM.git


Cloning into 'INDUSTRIAL-ENERGY-SIMULATION-SYSTEM'...


In [ ]:
%cd INDUSTRIAL-ENERGY-SIMULATION-SYSTEM

/content/INDUSTRIAL-ENERGY-SIMULATION-SYSTEM


In [ ]:
!ls



In [ ]:
!git add .


In [ ]:
!git commit -m "Updated project"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@ee61c23ea953.(none)')


In [ ]:
!git config --global user.email "isaacmugiraneza41@gmail.com"

In [ ]:
!git commit -m "Updated project"

On branch main

Initial commit

nothing to commit (create/copy files and use "git add" to track)


In [ ]:
!git add .

In [ ]:
!git commit -m "Updated project"

On branch main

Initial commit

nothing to commit (create/copy files and use "git add" to track)


In [ ]:
!git push origin main

error: src refspec main does not match any
error: failed to push some refs to 'https://github.com/Mugiraneza-user/INDUSTRIAL-ENERGY-SIMULATION-SYSTEM.git'


In [ ]:
!git remote add origin https://github.com/Mugiraneza-user/INDUSTRIAL-ENERGY-SIMULATION-SYSTEM.git

error: remote origin already exists.


In [ ]:
!git init

Reinitialized existing Git repository in /content/INDUSTRIAL-ENERGY-SIMULATION-SYSTEM/.git/


In [ ]:
!git remote -v

origin	https://github.com/Mugiraneza-user/INDUSTRIAL-ENERGY-SIMULATION-SYSTEM.git (fetch)
origin	https://github.com/Mugiraneza-user/INDUSTRIAL-ENERGY-SIMULATION-SYSTEM.git (push)


In [ ]:
!git branch